# 📏 Measuring Structural Truth: The RPF Score
### Validating Protein Structures against NMR Restraints

---

## 🎯 Objectives

In this tutorial, you will learn how to:
1.  **Define "Structural Truth"** in the context of solution NMR.
2.  **Calculate RPF Scores** (Recall, Precision, and F-measure) for a structural model.
3.  **Identify Over-fitting** by observing drops in the Precision score.
4.  **Benchmark Refinement** using real-world validation metrics from the Montelione group.

## 🧬 Background: What are RPF Scores?

NMR structure determination relies on **distance restraints** derived from the Nuclear Overhauser Effect (NOE). However, a structural model can look "perfect" geometrically (no clashes, good angles) but completely fail to represent the experimental data.

To bridge this gap, the **Montelione group** developed RPF scores (*Huang et al. 2005, JACS*):

- **Recall (R)**: Of the restraints we observed, how many are satisfied by the model? (Sensitivity)
- **Precision (P)**: Of the short distances in our model, how many are supported by our data? (Specificity)
- **F-measure (F)**: The harmonic mean of Recall and Precision. This is our single "Global Quality Score."

In [ ]:
import numpy as np
import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import matplotlib.pyplot as plt

from synth_pdb.generator import generate_pdb_content
from synth_pdb.nmr import calculate_rpf_score, calculate_synthetic_noes
from synth_pdb.validator import PDBValidator

print("✅ NMR Validation module loaded!")

## 🟢 Step 1: Generate a "Ground Truth" Structure

We will start by generating a high-quality, energy-minimized structure of a small protein domain and "observing" its NOEs.

In [ ]:
import io
# Generate a high-quality helix
seq = "AKAAKAKAAK" * 2
pdb_content = generate_pdb_content(sequence_str=seq, minimize_energy=True)

# Parse with Biotite
structure = pdb.PDBFile.read(io.StringIO(pdb_content)).get_structure(model=1)

# Generate "Experimental" NOEs from this ground truth
restraints = calculate_synthetic_noes(structure)
print(f"Generated {len(restraints)} NOE restraints from Ground Truth.")

## 🏅 Step 2: Validating the Perfect Fit

If we test the Ground Truth structure against its own NOEs, all scores should be **1.0**.

In [ ]:
scores = calculate_rpf_score(structure, restraints)
print("--- Ground Truth Scores ---")
for k, v in scores.items():
    print(f"{k.capitalize()}: {v:.4f}")

## ⚠️ Step 3: Identifying the "Over-folded" Decoy

What happens if we "compact" the protein artificially? The Recall will remain high (as the original contacts are still there), but the **Precision** will plummet because we have created many "new" short distances that are NOT supported by the data.

In [ ]:
# Create a 'collapsed' version by scaling coordinates towards the center
coords = structure.coord
center = np.mean(coords, axis=0)
collapsed_coords = (coords - center) * 0.7 + center # 30% collapse

collapsed_structure = structure.copy()
collapsed_structure.coord = collapsed_coords

collapsed_scores = calculate_rpf_score(collapsed_structure, restraints)
print("--- Collapsed Model Scores ---")
for k, v in collapsed_scores.items():
    print(f"{k.capitalize()}: {v:.4f}")

## 🚀 Summary

The RPF score is a powerful way to detect **over-fitting**. In real-world structure determination, we aim to maximize the F-measure. 

- **High Recall, Low Precision**? Your protein is likely too compact or "misfolded" locally.
- **Low Recall, High Precision**? You haven't satisfied your experimental constraints yet.